# Agentic System Showcase

This notebook is a scenario-first interactive walkthrough of the live `agentic_chatbot_next` system.
It starts the real FastAPI gateway, calls the real public API, and then reads runtime traces from disk so you can see what happened.

After setup and startup, the main path becomes a balanced intro followed by self-contained demo cells.
Each demo has a short scenario description directly above it, prints live summarized observability logs while it runs, and then renders structured post-run summaries for progress, tools, artifacts, metadata, and traces.
Use this source notebook for the interactive walkthrough; the executed notebook under `.artifacts/executed/` is only a smoke-test artifact.


## What This Notebook Covers

By the end of this walkthrough you will have seen:

- how the notebook drives the live backend through the public API
- how the setup and driver cells prepare the runtime before the demo sequence starts
- how the live `basic`, `general`, `rag_worker`, `data_analyst`, and `coordinator` roles behave in practice
- how current runtime docs from `docs/` back the grounded RAG and long-form demos
- how inline observability looks inside a normal demo cell: summarized progress, tools, artifacts, metadata, and traces
- how workbook-backed and upload-backed demos can be run independently
- how repo-authored skill packs differ from runtime-authored skills managed through `/v1/skills`

Important note: this notebook shows **summarized execution progress**, not raw chain-of-thought.
The streaming events are safe operational signals such as route decisions, tool intents, worker activity, evidence progress, long-form phase updates, and handoff milestones.


## How To Use This Notebook

1. Run the setup cells first so the notebook can validate the live runtime configuration.
2. If `START_LOCAL_SERVER=True`, the notebook will try to bootstrap missing local services such as `rag-postgres` and build the offline analyst sandbox image if needed, rerun strict preflight, and then start the live backend.
3. After setup succeeds, the main path moves through standalone demos where each runnable chat cell sends one user query and each skill control cell performs one runtime skill action.
4. Each chat demo cell prepares its own ingest or upload requirements before running, so you can execute only the sections you care about.
5. The runtime skill control walkthrough is intentionally sequential: run those cells from top to bottom after the repo skill-pack overview.
6. Open this source notebook for the walkthrough; the executed notebook artifact under `.artifacts/executed/` is only a test snapshot.


In [ ]:
from pathlib import Path

BASE_URL = ''
START_LOCAL_SERVER = True
SERVER_READINESS_PATH = '/health/live'
GATEWAY_TIMEOUT_SECONDS = 1800
JOB_WAIT_TIMEOUT_SECONDS = 900

RUN_ADVANCED_DEMOS = True
RUN_DEFENSE_CORPUS_DEMO = False
SHOW_SERVER_LOG_TAIL = False

NOTEBOOK_TENANT_ID = 'local-dev'
NOTEBOOK_USER_ID = 'notebook-demo'

DEMO_COLLECTION_ID = 'notebook-demo'
DEFENSE_COLLECTION_ID = 'defense-rag-test'

CUSTOM_INGEST_PATHS = []
DEFENSE_CORPUS_ROOT = 'defense_rag_test_corpus/documents'
DEFAULT_DEMO_PATHS = [
    'docs/ARCHITECTURE.md',
    'docs/CONTROL_FLOW.md',
    'docs/OPENAI_GATEWAY.md',
    'docs/TOOLS_AND_TOOL_CALLING.md',
    'docs/SKILLS_PLAYBOOK.md',
    'docs/DATA_ANALYST_AGENT.md',
    'docs/WORKSPACE.md',
]
WORKBOOK_DEMO_PATH = 'new_demo_notebook/demo_data/data_analyst/sales_performance.xlsx'
DATA_ANALYST_DEMO_PATHS = [
    'new_demo_notebook/demo_data/regional_spend.csv',
    'new_demo_notebook/demo_data/regional_controls.csv',
]


## Environment And Imports

The notebook works from the repository root or from the `new_demo_notebook/` folder.
This setup cell resolves the repo root, imports the notebook helper modules, and prepares the runtime paths used throughout the walkthrough.


In [ ]:
import json
import os
import re
import sys
import time
from pathlib import Path
from pprint import pprint

import httpx
from IPython.display import Markdown, JSON, display

CWD = Path.cwd().resolve()
if (CWD / 'run.py').exists():
    REPO_ROOT = CWD
elif (CWD.parent / 'run.py').exists():
    REPO_ROOT = CWD.parent
else:
    raise RuntimeError('Could not locate repository root from the current working directory.')

SRC_ROOT = REPO_ROOT / 'src'
NOTEBOOK_ROOT = REPO_ROOT / 'new_demo_notebook'
ARTIFACTS_DIR = NOTEBOOK_ROOT / '.artifacts'
RUNTIME_ROOT = REPO_ROOT / 'data' / 'runtime'
WORKSPACE_ROOT = REPO_ROOT / 'data' / 'workspaces'
MEMORY_ROOT = REPO_ROOT / 'data' / 'memory'
NOTEBOOK_PATH = NOTEBOOK_ROOT / 'agentic_system_showcase.ipynb'


for candidate in (str(REPO_ROOT), str(SRC_ROOT)):
    if candidate not in sys.path:
        sys.path.insert(0, candidate)

from new_demo_notebook.lib.client import GatewayClient, GatewayStreamResult
import new_demo_notebook.lib.preflight as notebook_preflight_module
from new_demo_notebook.lib.preflight import bootstrap_local_dependencies, run_preflight
from new_demo_notebook.lib.render import (
    build_workspace_preview,
    display_stream_result,
    display_trace_bundle,
    stream_event_console_line,
)
from new_demo_notebook.lib.server import BackendServerManager
from new_demo_notebook.lib.trace_reader import (
    cleanup_conversation_artifacts,
    collect_trace_bundle,
    extract_observed_agents,
    extract_observed_event_types,
    extract_observed_route,
)

PREFLIGHT_HELPER_PATH = Path(notebook_preflight_module.__file__).resolve()

server = None
client = None
model_id = ''


## How The Notebook Driver Works

The notebook is intentionally API-driven.
It does **not** call private runtime internals directly for demos.

The main moving parts are:

1. `BackendServerManager`: starts and stops the live FastAPI server.
2. `GatewayClient`: calls `/v1/chat/completions`, `/v1/ingest/documents`, and `/v1/skills`, including streaming SSE support for progress, artifacts, and response metadata.
3. `render.py`: groups live progress events into a readable thought timeline, tool log view, returned artifact list, response metadata view, and trace summary.
4. `trace_reader.py`: reads persisted runtime sessions, events, jobs, and workspaces after each demo.
5. `scenario_runner.py`: remains available as a separate acceptance harness for broad coverage validation, but the notebook itself stays hand-authored and scenario-first.

The next cell prints a compact source-level map of those helpers so you can connect the tutorial flow to the actual driver code.


In [ ]:
HELPER_COMPONENTS = [
    {
        'name': 'BackendServerManager',
        'path': REPO_ROOT / 'new_demo_notebook' / 'lib' / 'server.py',
        'purpose': 'Starts the live FastAPI backend and waits for a health endpoint.',
    },
    {
        'name': 'GatewayClient',
        'path': REPO_ROOT / 'new_demo_notebook' / 'lib' / 'client.py',
        'purpose': 'Calls the gateway, including streaming chat, ingest, skill CRUD, artifacts, and metadata events.',
    },
    {
        'name': 'Render Helpers',
        'path': REPO_ROOT / 'new_demo_notebook' / 'lib' / 'render.py',
        'purpose': 'Turns progress events, response metadata, artifacts, and traces into notebook-friendly views.',
    },
    {
        'name': 'Trace Reader',
        'path': REPO_ROOT / 'new_demo_notebook' / 'lib' / 'trace_reader.py',
        'purpose': 'Loads persisted sessions, jobs, events, transcripts, and workspace files.',
    },
    {
        'name': 'Scenario Runner',
        'path': REPO_ROOT / 'new_demo_notebook' / 'lib' / 'scenario_runner.py',
        'purpose': 'Runs the separate acceptance scenario manifest and validates broad agent coverage outside the notebook walkthrough.',
    },
]

def read_excerpt(path: Path, *, max_lines: int = 18) -> str:
    lines = path.read_text(encoding='utf-8').splitlines()
    excerpt = "\n".join(lines[:max_lines])
    if len(lines) > max_lines:
        excerpt += "\n..."
    return excerpt

display(Markdown('### Driver Helper Map'))
display(JSON([{k: str(v) if isinstance(v, Path) else v for k, v in item.items()} for item in HELPER_COMPONENTS]))

for item in HELPER_COMPONENTS:
    display(Markdown(f"#### {item['name']} — `{item['path'].relative_to(REPO_ROOT)}`"))
    display(Markdown(item['purpose']))
    print(read_excerpt(item['path']))
    print("\n" + '=' * 100 + "\n")


## Notebook Helper Functions

This utility cell defines the reusable scenario helpers used below.
They keep the scenario cells short and consistent while still surfacing live logs, prep actions, returned artifacts, response metadata, workspace previews, and persisted traces.


In [ ]:
TERMINAL_JOB_STATUSES = {'completed', 'failed', 'stopped'}


def md(text: str) -> None:
    display(Markdown(text))


def pretty(value) -> None:
    display(JSON(value))


def resolve_repo_path(raw: str | Path) -> Path:
    path = Path(raw).expanduser()
    return path if path.is_absolute() else (REPO_ROOT / path)


def existing_repo_paths(raw_paths) -> list[str]:
    resolved = []
    for raw in raw_paths:
        path = resolve_repo_path(raw)
        if path.exists():
            resolved.append(str(path))
    return resolved


def conversation_id_for(slug: str) -> str:
    safe = re.sub(r'[^a-z0-9-]+', '-', slug.lower()).strip('-')
    return f'notebook-{safe}'


def collection_id_for(slug: str) -> str:
    safe = re.sub(r'[^a-z0-9-]+', '-', slug.lower()).strip('-')
    return f'{DEMO_COLLECTION_ID}-{safe}'


def safe_health_ready(base_url: str) -> dict:
    try:
        response = httpx.get(f"{base_url.rstrip('/')}/health/ready", timeout=10.0)
        payload = response.json() if response.content else {}
        return {'status_code': response.status_code, 'payload': payload}
    except Exception as exc:
        return {'status_code': 'error', 'payload': {'error': str(exc)}}


def print_stream_event(event) -> None:
    line = stream_event_console_line(event)
    if event.kind == 'content':
        print(line, end='', flush=True)
    else:
        print(line, flush=True)


def merge_stream_results(results: list[GatewayStreamResult]) -> GatewayStreamResult:
    merged_metadata = {}
    for item in results:
        merged_metadata.update(item.metadata)
    return GatewayStreamResult(
        text='\n\n'.join(item.text for item in results if item.text).strip(),
        events=[event for item in results for event in item.events],
        progress_events=[event for item in results for event in item.progress_events],
        artifacts=[artifact for item in results for artifact in item.artifacts],
        metadata=merged_metadata,
        raw_chunks=[chunk for item in results for chunk in item.raw_chunks],
    )


def wait_for_terminal_jobs(conversation_id: str, *, timeout_seconds: float = JOB_WAIT_TIMEOUT_SECONDS):
    deadline = time.monotonic() + timeout_seconds
    latest_bundle = collect_trace_bundle(RUNTIME_ROOT, WORKSPACE_ROOT, conversation_id)
    while time.monotonic() < deadline:
        latest_bundle = collect_trace_bundle(RUNTIME_ROOT, WORKSPACE_ROOT, conversation_id)
        if not latest_bundle.jobs or all(str(job.get('status') or '') in TERMINAL_JOB_STATUSES for job in latest_bundle.jobs):
            return latest_bundle
        time.sleep(0.25)
    return latest_bundle


def trace_summary(bundle) -> dict:
    return {
        'conversation_id': bundle.conversation_id,
        'observed_route': extract_observed_route(bundle),
        'observed_agents': extract_observed_agents(bundle),
        'observed_event_types': extract_observed_event_types(bundle),
        'job_count': len(bundle.jobs),
        'notification_count': len(bundle.notifications),
        'workspace_files': bundle.workspace_files,
    }


def cleanup_demo_conversation(conversation_id: str) -> None:
    cleanup_conversation_artifacts(RUNTIME_ROOT, WORKSPACE_ROOT, conversation_id)


def stream_turn(
    *,
    conversation_id: str,
    prompt: str,
    history: list[dict],
    force_agent: bool = False,
    collection_id: str = '',
    metadata: dict | None = None,
) -> GatewayStreamResult:
    assert client is not None, 'Client has not been initialized yet.'
    events = []
    for event in client.stream_chat_turn(
        history=history,
        user_text=prompt,
        conversation_id=conversation_id,
        model=model_id,
        force_agent=force_agent,
        metadata=metadata or {},
        tenant_id=NOTEBOOK_TENANT_ID,
        user_id=NOTEBOOK_USER_ID,
        collection_id=collection_id,
    ):
        events.append(event)
        print_stream_event(event)
    print('\n')
    return GatewayClient.collect_stream(events)


def ingest_demo_paths(
    *,
    paths: list[str],
    conversation_id: str,
    collection_id: str,
    source_type: str = 'upload',
) -> dict:
    assert client is not None, 'Client has not been initialized yet.'
    resolved = existing_repo_paths(paths)
    if not resolved:
        raise FileNotFoundError(f'No ingest paths were found. Requested: {paths}')
    response = client.ingest(
        paths=resolved,
        conversation_id=conversation_id,
        source_type=source_type,
        collection_id=collection_id,
        tenant_id=NOTEBOOK_TENANT_ID,
        user_id=NOTEBOOK_USER_ID,
    )
    pretty({'collection_id': collection_id, 'paths': resolved, 'response': response})
    return response


def prepare_collection(*, paths: list[str], conversation_id: str, collection_id: str) -> dict:
    md('### Collection Prep')
    return ingest_demo_paths(
        paths=paths,
        conversation_id=conversation_id,
        collection_id=collection_id,
        source_type='upload',
    )


def prepare_uploads(*, paths: list[str], conversation_id: str, collection_id: str) -> dict:
    md('### Upload Prep')
    return ingest_demo_paths(
        paths=paths,
        conversation_id=conversation_id,
        collection_id=collection_id,
        source_type='upload',
    )


def build_reference_packet(paths: list[str], *, max_chars_per_file: int = 1200) -> str:
    parts = []
    for raw in paths:
        path = resolve_repo_path(raw)
        if not path.exists() or not path.is_file():
            continue
        text = path.read_text(encoding='utf-8')[:max_chars_per_file].strip()
        if not text:
            continue
        if path.stat().st_size > max_chars_per_file:
            text += '\n...'
        parts.append(f'SOURCE: {path.name}\n{text}')
    return '\n\n'.join(parts)


def show_workspace_preview(bundle, filename: str, *, title: str, max_chars: int = 1200) -> None:
    preview = build_workspace_preview(bundle, filename, max_chars=max_chars)
    if not preview:
        print(f'No workspace preview available for {filename}.')
        return
    md(f'### {title}')
    pretty(preview)


def show_long_output_previews(result: GatewayStreamResult, bundle, *, max_chars: int = 1200) -> None:
    long_output = dict(result.metadata.get('long_output') or {})
    if not long_output:
        return
    output_filename = str(long_output.get('output_filename') or '')
    manifest_filename = str(long_output.get('manifest_filename') or '')
    if output_filename:
        show_workspace_preview(bundle, output_filename, title='Long-Form Draft Preview', max_chars=max_chars)
    if manifest_filename:
        show_workspace_preview(bundle, manifest_filename, title='Long-Form Manifest Preview', max_chars=max_chars)


def run_notebook_scenario(
    title: str,
    *,
    conversation_id: str,
    prompt: str,
    force_agent: bool = False,
    collection_id: str = '',
    trace_focus: list[str] | None = None,
    metadata: dict | None = None,
    collection_paths: list[str] | None = None,
    upload_paths: list[str] | None = None,
    show_long_output: bool = False,
):
    cleanup_demo_conversation(conversation_id)
    if collection_paths:
        prepare_collection(
            paths=list(collection_paths),
            conversation_id=conversation_id,
            collection_id=collection_id,
        )
    if upload_paths:
        prepare_uploads(
            paths=list(upload_paths),
            conversation_id=conversation_id,
            collection_id=collection_id,
        )
    md(f'### {title}')
    print(f'Conversation: {conversation_id}')
    if collection_id:
        print(f'Collection: {collection_id}')
    result = stream_turn(
        conversation_id=conversation_id,
        prompt=prompt,
        history=[],
        force_agent=force_agent,
        collection_id=collection_id,
        metadata=metadata,
    )
    bundle = wait_for_terminal_jobs(conversation_id)
    pretty(trace_summary(bundle))
    display_stream_result(result, title=title, trace_bundle=bundle, trace_focus=trace_focus)
    if show_long_output:
        show_long_output_previews(result, bundle)
    return result, bundle


def list_agent_catalog() -> list[dict]:
    rows = []
    for path in sorted((REPO_ROOT / 'data' / 'agents').glob('*.md')):
        text = path.read_text(encoding='utf-8')
        parts = text.split('---', 2)
        if len(parts) < 3:
            continue
        frontmatter = parts[1]
        fields = {}
        for line in frontmatter.splitlines():
            if ':' not in line:
                continue
            key, value = line.split(':', 1)
            fields[key.strip()] = value.strip()
        rows.append(
            {
                'name': fields.get('name', path.stem),
                'mode': fields.get('mode', ''),
                'description': fields.get('description', ''),
                'skill_scope': fields.get('skill_scope', ''),
                'allowed_tools': fields.get('allowed_tools', ''),
                'allowed_worker_agents': fields.get('allowed_worker_agents', ''),
                'path': str(path.relative_to(REPO_ROOT)),
            }
        )
    return rows


def skill_pack_template(skill_name: str = 'Notebook Runtime Research Skill') -> str:
    return f"""# {skill_name}
agent_scope: rag
tool_tags: search_collection, search_all_documents, fetch_chunk_window
task_tags: process flow, corpus discovery, grounded research
description: Example skill pack created from the showcase notebook.
retrieval_profile: process_flow_identification
controller_hints: {{"prefer_keyword_window": true, "prefer_doc_inventory": true, "prefer_process_flow": true}}
coverage_goal: corpus_wide
result_mode: inventory
enabled: false

## When To Use
Use this skill when the user wants a grounded inventory of routing, handoff, or process-flow evidence across repo docs.

## Guidance
Start broad, resolve the strongest matching docs, then reread nearby windows before writing a synthesis.
"""


def show_server_log_tail(max_chars: int = 4000) -> None:
    if server is None:
        print('No local server was started in this notebook session.')
        return
    tail = server.read_log_tail(max_chars=max_chars)
    print(tail or '(server log is empty)')


def cleanup_resources(force: bool = True) -> None:
    global client, server
    if client is not None:
        client.close()
        client = None
    if server is not None:
        server.stop(force=force)
        server = None


## Environment Preflight

This preflight uses the same runtime settings loader as the live backend, so the notebook checks the real `.env`-resolved database, provider, and model configuration.
When `START_LOCAL_SERVER=True`, the next cell will try to bootstrap missing local Docker dependencies such as `rag-postgres` and the offline analyst sandbox image, rerun the strict checks, and then fail hard if the environment is still not runnable.


In [ ]:
bootstrap_actions = []
preflight_report = run_preflight(
    repo_root=REPO_ROOT,
    runtime_root=RUNTIME_ROOT,
    workspace_root=WORKSPACE_ROOT,
    memory_root=MEMORY_ROOT,
)
if START_LOCAL_SERVER and not preflight_report.ready:
    bootstrap_actions = bootstrap_local_dependencies(repo_root=REPO_ROOT, report=preflight_report)
    if bootstrap_actions:
        preflight_report = run_preflight(
            repo_root=REPO_ROOT,
            runtime_root=RUNTIME_ROOT,
            workspace_root=WORKSPACE_ROOT,
            memory_root=MEMORY_ROOT,
        )
pretty({
    'notebook_path': str(NOTEBOOK_PATH.resolve()),
    'preflight_helper_path': str(PREFLIGHT_HELPER_PATH),
    'resolved_settings': preflight_report.resolved_settings,
    'checks': preflight_report.to_rows(),
    'blocking_checks': [
        {
            'name': check.name,
            'detail': check.detail,
            'hint': check.hint,
        }
        for check in preflight_report.blocking_checks()
    ],
    'failure_summary': preflight_report.failure_summary(),
    'bootstrap_actions': [action.to_row() for action in bootstrap_actions],
})
assert preflight_report.ready, f"Notebook preflight failed. {preflight_report.failure_summary()}"


## Start The Live Backend

By default the notebook starts `python run.py serve-api` for you after the strict preflight succeeds.
The notebook waits on `/health/live` so the walkthrough can start even if the KB is not yet fully indexed.
The next cell then inspects `/health/ready` separately so you can see whether the KB layer is already warm or whether you should ingest data into a collection first.


In [ ]:
os.environ['NEXT_RUNTIME_GATEWAY_TIMEOUT_SECONDS'] = str(GATEWAY_TIMEOUT_SECONDS)
os.environ['NEXT_RUNTIME_JOB_WAIT_SECONDS'] = str(JOB_WAIT_TIMEOUT_SECONDS)

if START_LOCAL_SERVER:
    server = BackendServerManager(
        repo_root=REPO_ROOT,
        artifacts_dir=ARTIFACTS_DIR,
        readiness_path=SERVER_READINESS_PATH,
    )
    BASE_URL = server.start()
elif not BASE_URL:
    raise RuntimeError('Set BASE_URL when START_LOCAL_SERVER is False.')

client = GatewayClient(BASE_URL, timeout_seconds=GATEWAY_TIMEOUT_SECONDS)
model_id = client.get_model_id()
print({'base_url': BASE_URL, 'model_id': model_id, 'readiness_path': SERVER_READINESS_PATH, 'gateway_timeout_seconds': GATEWAY_TIMEOUT_SECONDS, 'job_wait_timeout_seconds': JOB_WAIT_TIMEOUT_SECONDS})


## Health, Configuration, And Runtime Scope

This cell shows both liveness and readiness.
Readiness can be degraded before you ingest or sync a collection, which is expected if you are using this notebook to seed a brand-new demo collection.


In [ ]:
live_health = client.health_live()
ready_health = safe_health_ready(BASE_URL)

pretty(
    {
        'base_url': BASE_URL,
        'model_id': model_id,
        'live_health': live_health,
        'ready_health': ready_health,
        'tenant_id': NOTEBOOK_TENANT_ID,
        'user_id': NOTEBOOK_USER_ID,
        'demo_collection_prefix': DEMO_COLLECTION_ID,
        'run_advanced_demos': RUN_ADVANCED_DEMOS,
        'run_defense_corpus_demo': RUN_DEFENSE_CORPUS_DEMO,
    }
)


## Agent Catalog

The system exposes a mix of top-level agents and internal worker specialists.
Not every agent has a first-class public API entrypoint, but the notebook surfaces the user-facing agents directly and the internal worker agents through coordinator traces.


In [ ]:
agent_rows = list_agent_catalog()
pretty(agent_rows)


## Basic Chat Demo

- `User ask:` A simple greeting and capability question with no grounding request.
- `Expected runtime behavior:` The router should take the deterministic `BASIC` path and skip tools, workers, and retrieval.
- `Expected output:` A direct assistant reply plus minimal router/basic-turn trace events. Inline observability still appears here, but it should stay light.


In [ ]:
basic_result, basic_bundle = run_notebook_scenario(
    'Basic Chat Demo',
    conversation_id=conversation_id_for('basic-chat'),
    prompt='Hello there. Give me a brief explanation of what this assistant can help with.',
    force_agent=False,
    trace_focus=['basic', 'router_decision', 'basic_turn_started', 'basic_turn_completed'],
)


## General Agent Tool-And-Skill Demo

- `User ask:` Ask one question about which indexed docs and retrieved skill guidance would best support investigating routing, tool calling, and skill behavior in this runtime.
- `Expected runtime behavior:` This should stay on the `general` agent path and surface top-level tool usage such as `list_indexed_docs` and `search_skills` without escalating to `coordinator`.
- `Expected output:` A grouped doc inventory plus a concise explanation of the matched skills. No artifacts are expected, but the cell should show tool activity and post-run summaries.


In [ ]:
general_collection_id = collection_id_for('general-agent')
general_result, general_bundle = run_notebook_scenario(
    'General Agent Tool-And-Skill Demo',
    conversation_id=conversation_id_for('general-agent'),
    prompt=(
        'Which indexed documents in this collection are most relevant for investigating routing, tool calling, '
        'and skill behavior in this runtime, and what retrieved skill guidance would help search them?'
    ),
    force_agent=True,
    collection_id=general_collection_id,
    collection_paths=DEFAULT_DEMO_PATHS,
    trace_focus=['general', 'list_indexed_docs', 'search_skills', 'tool_start', 'tool_end'],
)


## Delegated Grounded RAG Demo

- `User ask:` Give a grounded explanation of how the OpenAI-compatible gateway scopes requests and supports `requested_agent` overrides.
- `Expected runtime behavior:` The router still chooses `AGENT`, but `metadata.requested_agent="general"` should keep the turn on the general-agent tool wrapper so you can see delegated RAG behavior.
- `Expected output:` A cited answer grounded in the indexed docs plus visible `rag_agent_tool`/tool-trace activity. No artifacts are expected.


In [ ]:
delegated_rag_collection_id = collection_id_for('delegated-grounded-rag')
delegated_rag_result, delegated_rag_bundle = run_notebook_scenario(
    'Delegated Grounded RAG Demo',
    conversation_id=conversation_id_for('delegated-grounded-rag'),
    prompt=(
        'Using only the indexed documents in this collection, explain how the OpenAI-compatible gateway scopes requests '
        'and supports requested-agent overrides. Cite your sources.'
    ),
    force_agent=True,
    collection_id=delegated_rag_collection_id,
    collection_paths=DEFAULT_DEMO_PATHS,
    metadata={'requested_agent': 'general'},
    trace_focus=['general', 'rag_agent_tool', 'tool_start', 'tool_end'],
)


## Direct Grounded RAG Demo

- `User ask:` Explain how the gateway, router, and runtime service fit together, including the long-form branch.
- `Expected runtime behavior:` This should route directly into `rag_worker` rather than the general ReAct tool wrapper.
- `Expected output:` A grounded synthesis with citations from the indexed runtime docs and a simpler trace than the delegated demo, because top-level tool calls should not dominate the run.


In [ ]:
direct_rag_collection_id = collection_id_for('direct-grounded-rag')
rag_result, rag_bundle = run_notebook_scenario(
    'Direct Grounded RAG Demo',
    conversation_id=conversation_id_for('direct-grounded-rag'),
    prompt=(
        'Using only the indexed documents in this collection, explain how the gateway, router, and runtime service fit together, '
        'including long-form writing support. Cite your sources.'
    ),
    force_agent=False,
    collection_id=direct_rag_collection_id,
    collection_paths=DEFAULT_DEMO_PATHS,
    trace_focus=['rag_worker', 'router_decision', 'agent_turn_started', 'agent_turn_completed'],
)


## Workbook-Aware RAG Demo

- `User ask:` Answer a grounded question against an indexed Excel workbook.
- `Expected runtime behavior:` This optional demo should use the direct grounded path against workbook-backed content after the cell ingests the workbook into its own collection.
- `Expected output:` A concise cited answer that names the workbook location. No artifacts are expected; the main goal is to prove workbook-aware grounding works independently of the other demos.


In [ ]:
if RUN_ADVANCED_DEMOS:
    workbook_collection_id = collection_id_for('workbook-rag')
    workbook_result, workbook_bundle = run_notebook_scenario(
        'Workbook-Aware RAG Demo',
        conversation_id=conversation_id_for('workbook-rag'),
        prompt=(
            'In the indexed sales_performance workbook, which region has the highest monthly target revenue '
            'and what is the target amount? Cite the workbook location.'
        ),
        force_agent=False,
        collection_id=workbook_collection_id,
        collection_paths=[WORKBOOK_DEMO_PATH],
        trace_focus=['rag_worker', 'router_decision', 'agent_turn_started', 'agent_turn_completed'],
    )
else:
    print('Skipping workbook-aware RAG demo because RUN_ADVANCED_DEMOS is False.')


## Data Analyst Sandbox Demo

- `User ask:` Join the uploaded CSV files, calculate reserve gaps, and return the derived file.
- `Expected runtime behavior:` The router should pick `data_analyst`, then the agent should inspect the data, execute Python inside the offline sandbox, and publish the result with `return_file`.
- `Expected output:` A concise analysis summary plus a downloadable artifact in the top-level `artifacts` array. Trace output should highlight `load_dataset`, `inspect_columns`, `execute_code`, and `return_file`.


In [ ]:
analyst_collection_id = collection_id_for('data-analyst')
analyst_result, analyst_bundle = run_notebook_scenario(
    'Data Analyst Sandbox Demo',
    conversation_id=conversation_id_for('data-analyst'),
    prompt=(
        'Analyze the uploaded CSV files. Join them on region, calculate reserve_gap_usd as reserve_target_usd minus monthly_spend_usd, '
        'rank the top three regions by reserve gap, write the joined result to a CSV, and return the file.'
    ),
    force_agent=False,
    collection_id=analyst_collection_id,
    upload_paths=DATA_ANALYST_DEMO_PATHS,
    trace_focus=['data_analyst', 'load_dataset', 'inspect_columns', 'execute_code', 'return_file'],
)


## Coordinator Orchestration Demo

- `User ask:` Combine sandboxed spreadsheet analysis with grounded doc research, then synthesize and verify one operator-ready answer.
- `Expected runtime behavior:` This should trigger `coordinator` orchestration with planner, worker execution, handoff events, finalization, and verification.
- `Expected output:` A cited multi-step answer with rich trace detail showing the worker lifecycle. The emphasis is on coordination and verification rather than returning a file.


In [ ]:
coordinator_collection_id = collection_id_for('coordinator-orchestration')
coordinator_result, coordinator_bundle = run_notebook_scenario(
    'Coordinator Orchestration Demo',
    conversation_id=conversation_id_for('coordinator-orchestration'),
    prompt=(
        'Coordinate this as a multi-step workflow. First analyze the uploaded CSV files to identify the three largest reserve gaps. '
        'Then search the indexed runtime docs for the data analyst workspace model, copy-on-write output rules, and artifact exposure path. '
        'Pass the structured findings between workers, synthesize an operator-ready summary, and verify the final answer with citations.'
    ),
    force_agent=False,
    collection_id=coordinator_collection_id,
    collection_paths=DEFAULT_DEMO_PATHS,
    upload_paths=DATA_ANALYST_DEMO_PATHS,
    trace_focus=['coordinator', 'planner', 'data_analyst', 'rag_worker', 'finalizer', 'verifier', 'handoff_prepared', 'handoff_consumed'],
)


## Long-Form Writing Demo

- `User ask:` Produce a four-section operator briefing grounded in the current runtime docs and attach the full draft.
- `Expected runtime behavior:` The turn should stay on `general` with `metadata.long_output` enabled, so the long-form composer writes a workspace-backed draft and returns only a short summary inline.
- `Expected output:` Assistant summary text, `metadata.long_output`, a downloadable Markdown artifact, and workspace previews of the generated draft and manifest.


In [ ]:
long_form_collection_id = collection_id_for('long-form-writing')
long_form_reference_packet = build_reference_packet(DEFAULT_DEMO_PATHS, max_chars_per_file=900)
long_form_prompt = (
    'Write a four-section operator briefing for someone new to this runtime. '
    'Cover routing and requested-agent overrides, grounded retrieval, sandboxed data analysis, and runtime skill control. '
    'Keep the writing concrete and attach the full Markdown draft as a downloadable artifact.\n\n'
    'REFERENCE_CONTEXT:\n'
    f'{long_form_reference_packet}'
)
long_form_result, long_form_bundle = run_notebook_scenario(
    'Long-Form Writing Demo',
    conversation_id=conversation_id_for('long-form-writing'),
    prompt=long_form_prompt,
    force_agent=True,
    collection_id=long_form_collection_id,
    collection_paths=DEFAULT_DEMO_PATHS,
    metadata={
        'requested_agent': 'general',
        'long_output': {
            'enabled': True,
            'target_words': 1800,
            'target_sections': 4,
            'delivery_mode': 'hybrid',
            'output_format': 'markdown',
        },
    },
    trace_focus=['general', 'agent_turn_started', 'agent_turn_completed'],
    show_long_output=True,
)


## Repo Skill-Pack Overview

- `User ask:` Inspect the repo-authored skill-pack surface that feeds runtime retrieval and prompt guidance.
- `Expected runtime behavior:` This is a local notebook inspection step, not a chat turn. It should surface the available skill scopes and representative checked-in packs.
- `Expected output:` The skill-pack root, available scopes, a small sample of repo-authored packs, and a template aligned with the current runtime skill metadata shape.


In [ ]:
skill_root = REPO_ROOT / 'data' / 'skill_packs'
pretty(
    {
        'skill_root': str(skill_root),
        'available_scopes': sorted(path.name for path in skill_root.iterdir() if path.is_dir()),
        'sample_skill_packs': {
            path.name: sorted(child.stem for child in path.glob('*.md'))[:4]
            for path in sorted(skill_root.iterdir())
            if path.is_dir()
        },
        'example_template': skill_pack_template(),
    }
)


## Create Runtime Skill Draft

- `Action:` Create one private draft runtime skill and initialize the shared notebook variables used by the later skill-control cells.
- `Expected runtime behavior:` This calls `/v1/skills` once to create the draft record with notebook-scoped tenant and user headers.
- `Expected output:` The created skill payload. Run this cell first before the later skill lifecycle cells.


In [ ]:
skill_id = 'notebook-runtime-routing-skill'
skill_body_v1 = skill_pack_template('Notebook Runtime Routing Skill')
created_skill = client.create_skill(
    {
        'skill_id': skill_id,
        'body_markdown': skill_body_v1,
        'visibility': 'private',
        'status': 'draft',
        'owner_user_id': NOTEBOOK_USER_ID,
    },
    tenant_id=NOTEBOOK_TENANT_ID,
    user_id=NOTEBOOK_USER_ID,
)
pretty(created_skill)


## Preview Runtime Skill Resolution

- `Action:` Preview how the draft skill would resolve for a routing/process-flow query without mutating live state.
- `Expected runtime behavior:` This calls `/v1/skills/preview` once.
- `Expected output:` A preview payload showing the current retrieval result. Run the create step first.


In [ ]:
assert 'skill_id' in globals() and skill_id, 'Run Create Runtime Skill Draft first.'
preview_before = client.preview_skill_search(
    query='routing handoff process flow',
    agent_scope='rag',
    tenant_id=NOTEBOOK_TENANT_ID,
    user_id=NOTEBOOK_USER_ID,
)
pretty(preview_before)


## Activate Runtime Skill

- `Action:` Activate the runtime skill draft created above.
- `Expected runtime behavior:` This calls `/v1/skills/{skill_id}/activate` once.
- `Expected output:` The activation response payload. Run the create step first.


In [ ]:
assert 'skill_id' in globals() and skill_id, 'Run Create Runtime Skill Draft first.'
activated_skill = client.activate_skill(
    skill_id,
    tenant_id=NOTEBOOK_TENANT_ID,
    user_id=NOTEBOOK_USER_ID,
)
pretty(activated_skill)


## Inspect Runtime Skill

- `Action:` Read back the current runtime skill record.
- `Expected runtime behavior:` This calls `GET /v1/skills/{skill_id}` once.
- `Expected output:` The inspected skill payload. Run the create step first.


In [ ]:
assert 'skill_id' in globals() and skill_id, 'Run Create Runtime Skill Draft first.'
inspected_skill = client.get_skill(
    skill_id,
    tenant_id=NOTEBOOK_TENANT_ID,
    user_id=NOTEBOOK_USER_ID,
)
pretty(inspected_skill)


## Update Runtime Skill Version

- `Action:` Submit one updated version of the runtime skill with slightly expanded guidance.
- `Expected runtime behavior:` This calls `PUT /v1/skills/{skill_id}` once and may return a versioned skill id for the updated record.
- `Expected output:` The updated skill payload plus the notebook variable used by the later deactivate and rollback cells. Run the create step first.


In [ ]:
assert 'skill_id' in globals() and skill_id, 'Run Create Runtime Skill Draft first.'
skill_body_v2 = skill_body_v1.replace(
    'routing, handoff, or process-flow evidence',
    'routing, handoff, artifact, or process-flow evidence',
)
updated_skill = client.update_skill(
    skill_id,
    {
        'body_markdown': skill_body_v2,
        'owner_user_id': NOTEBOOK_USER_ID,
        'visibility': 'private',
    },
    tenant_id=NOTEBOOK_TENANT_ID,
    user_id=NOTEBOOK_USER_ID,
)
updated_skill_id = updated_skill['data']['skill_id']
pretty(updated_skill)


## Deactivate Updated Runtime Skill

- `Action:` Deactivate the updated skill record returned by the update step.
- `Expected runtime behavior:` This calls `/v1/skills/{updated_skill_id}/deactivate` once.
- `Expected output:` The deactivation response payload. Run the update step first.


In [ ]:
assert 'updated_skill_id' in globals() and updated_skill_id, 'Run Update Runtime Skill Version first.'
deactivated_skill = client.deactivate_skill(
    updated_skill_id,
    tenant_id=NOTEBOOK_TENANT_ID,
    user_id=NOTEBOOK_USER_ID,
)
pretty(deactivated_skill)


## Roll Back Runtime Skill

- `Action:` Roll the updated skill record back to the original skill family created earlier.
- `Expected runtime behavior:` This calls `/v1/skills/{updated_skill_id}/rollback` once.
- `Expected output:` The rollback response payload. Run the create and update steps first.


In [ ]:
assert 'skill_id' in globals() and skill_id, 'Run Create Runtime Skill Draft first.'
assert 'updated_skill_id' in globals() and updated_skill_id, 'Run Update Runtime Skill Version first.'
rolled_back_skill = client.rollback_skill(
    updated_skill_id,
    target_skill_id=skill_id,
    tenant_id=NOTEBOOK_TENANT_ID,
    user_id=NOTEBOOK_USER_ID,
)
pretty(rolled_back_skill)


## Optional Defense Corpus Appendix

This appendix is disabled by default because it is heavier than the core walkthrough.
If you enable it, the notebook ingests the defense test corpus into its own collection and runs a representative benchmark-style question.

For repeated benchmark work, the CLI commands are still the better interface:

- `python run.py sync-defense-corpus`
- `python run.py evaluate-defense-corpus --sync-first`


In [ ]:
defense_root = resolve_repo_path(DEFENSE_CORPUS_ROOT)
if RUN_DEFENSE_CORPUS_DEMO:
    if not defense_root.exists():
        raise FileNotFoundError(f'Defense corpus root not found: {defense_root}')
    defense_ingest_response = ingest_demo_paths(
        paths=[str(defense_root)],
        conversation_id=conversation_id_for('defense-corpus-seed'),
        collection_id=DEFENSE_COLLECTION_ID,
    )
    defense_result, defense_bundle = run_stream_demo(
        'Defense Corpus Demo',
        conversation_id=conversation_id_for('defense-corpus-question'),
        prompt='What is the approved current CDR date for Asterion? Include citations.',
        force_agent=False,
        collection_id=DEFENSE_COLLECTION_ID,
        trace_focus=['rag_worker', 'doc_focus', 'summary'],
    )
else:
    print('Skipping defense corpus appendix because RUN_DEFENSE_CORPUS_DEMO is False.')
    print(f'Configured defense root: {defense_root}')


## Troubleshooting

If a demo fails and you started the backend locally from this notebook, the next cell can print the recent server log tail.
Keep it disabled unless you need it, because the log can be noisy.


In [ ]:
if SHOW_SERVER_LOG_TAIL:
    show_server_log_tail()
else:
    print('Skipping server log tail because SHOW_SERVER_LOG_TAIL is False.')


## Cleanup

Run the next cell when you are done if you started the local backend from this notebook.
It closes the client and stops the subprocess cleanly.


In [ ]:
cleanup_resources(force=True)
print('Notebook resources cleaned up.')
